# ClinGuard-DiLoCo: Kaggle session 5 (gate G3: "teach the eyes")

Session 4 showed that the untrained Qwen2.5-VL-3B cannot tell an enlarged heart from a normal one: its vision
encoder (the "eyes") was never taught chest X-rays. This session trains **only the vision side** (LoRA on the
vision encoder and the connector, 6.9 M trainable numbers; language model frozen) on the training studies of
all three hospitals pooled, in two ways, and then runs the vision check on each:

| GPU | Run | The eyes are taught by | Then |
|---|---|---|---|
| 0 | `g3_eyes_report` | writing the full report (the normal training task) | development reports + vision check |
| 1 | `g3_eyes_findings` | choosing the right sentence of a pair, e.g. "The heart is enlarged." / "The heart size is normal." (sick and healthy balanced) | vision check |

Estimate: about 2.5 hours. Training uploads to the private HF repo every ~15 min with a lease; after a crash,
run the notebook again and each job continues where it stopped (finished jobs are skipped).

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run (important):** click **Save Version** (top right) -> choose **Save & Run All (Commit)** -> **Save**.
Then you can close the tab: a saved version keeps running for up to 12 h. Do **not** run the cells by hand in
the editor; that "Draft Session" is switched off when the tab is idle. When the version has finished, open it
from the notebook's version history and send the output of the last cell.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two GPUs do not race for it.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ClinGuard/src")
from huggingface_hub import snapshot_download
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch: on each GPU, train, then run the vision check on the trained model

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
os.makedirs("/kaggle/working/logs", exist_ok=True)
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config configs/experiment"
def check(name):
    return (f"python scripts/vision_check.py --hf-repo {HF_REPO} --name session5_{name} --variants "
            f"--configs configs/experiment/g3_eyes_{name}.yaml --work-dir /kaggle/working/runs")
queues = {0: [("train g3_eyes_report", f"{RUN}/g3_eyes_report.yaml"), ("vision check", check("report"))],
          1: [("train g3_eyes_findings", f"{RUN}/g3_eyes_findings.yaml"), ("vision check", check("findings"))]}
procs = {}
for gpu, jobs in queues.items():
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    # '&&': the vision check runs only if training finished
    procs[gpu] = subprocess.Popen(" && ".join(cmd for _, cmd in jobs), shell=True, stdout=log,
                                  stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", " -> ".join(label for label, _ in jobs))

## Wait, printing progress every 5 minutes

In [ ]:
import time
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines()
             if l.strip() and "Warning" not in l and "warn" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="\n   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Summary (self-contained: after a restart, run cell 2 and then this cell)

In [ ]:
import json, os, sys
from huggingface_hub import hf_hub_download
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")
import run_job as rj
import vision_check
def fetch(path):
    try:
        return hf_hub_download(HF_REPO, path, force_download=True)
    except Exception as e:
        print("missing:", path, type(e).__name__); return None
for n in ("g3_eyes_report", "g3_eyes_findings"):
    rid = rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
    lease = fetch(f"runs/{rid}/LEASE.json")
    print(f"{n:18s} {rid}  status: {json.load(open(lease))['status'] if lease else '-'}")
    r = fetch(f"runs/{rid}/rounds.jsonl")
    if r:
        rows = [json.loads(l) for l in open(r)]
        site = next(iter(rows[0]["train_loss"]))
        print("   train loss by round:", " ".join(f"{x['train_loss'][site]:.3f}" for x in rows),
              f"| {rows[-1]['cost'][site]['train_sec'] / 100:.2f} s/step")
# one table: the untrained model (session 4, fp16 vision) and the two trained versions
combined = None
s4 = fetch("diagnostics/vision_check_session4.json")
if s4:
    combined = json.load(open(s4))
    combined["results"] = {"untrained (session 4)": combined["results"]["nf4_visfp16"]}
for n in ("report", "findings"):
    v = fetch(f"diagnostics/vision_check_session5_{n}.json")
    if v:
        v = json.load(open(v))
        if combined is None:
            combined = dict(v, results={})
        combined["results"].update({f"eyes taught by {n}": r for r in v["results"].values()})
if combined:
    vision_check.print_report(combined)

**Send Tasbid / Claude:** the output of the last cell. The report run's development reports stay in the HF
repo; they are scored locally in the evaluation environment.